In [1]:
import pandas as pd
import numpy as np
from skbio.stats.composition import clr, multiplicative_replacement

ori = pd.read_csv('ResultsCibersortx/CIBERSORTx_Origin_Results.csv')
ori = ori.iloc[:,1:23]

syn = pd.read_csv('ResultsCibersortx/CIBERSORTx_gaussiancopula_42_Results.csv')
syn = syn.iloc[:,1:23]

In [2]:
import sys
parent_path = '/mnt/digphat/syntheticDataBenchmark/Chuong_pipeline'
sys.path.append(parent_path)
from SynOmics.metrics.fidelity.UnivariateSimilarity import UnivariateSimilarity
from SynOmics.processing.metadata import MetaData

In [ ]:
def log_center(df):
    orig_filled = multiplicative_replacement(df.values)
    clr_orig = clr(orig_filled)
    
    clr_orig_df = pd.DataFrame(clr_orig, index=df.index, columns=df.columns)

    return clr_orig_df



clr_ori = log_center(ori)


metadata = MetaData.get_metadata(data = clr_ori, 
                                 threshold_unique_values = 10, 
                                    ordinal_features = None)

results_all = {}
seeds = [0,1,2,3,42]
for seed in seeds:
    print(f"---Seed {seed}---")
    syn_datas = [f"avatarsk5_{seed}",f"avatarsk10_{seed}",f"ctgan_{seed}",f"gaussiancopula_{seed}",f"synthpop_{seed}",f"tvae_{seed}"]
    results_seed = {}
    for syn_data in syn_datas:
        try:
            print(f"---Method {syn_data}---")
            syn_df = pd.read_csv(f'ResultsCibersortx/CIBERSORTx_{syn_data}_Results.csv')
            syn = syn_df.iloc[:,1:23]
            clr_syn = log_center(syn)
            uni_comparison = UnivariateSimilarity(output_dir = f"KSStatistic/Seed_{seed}", logger_name = syn_data)
            scores = uni_comparison.get_univariate_score(
                                        original_data = clr_ori, 
                                        synthetic_data=clr_syn, 
                                        metadata=metadata)
            
            scores_df = uni_comparison.get_detail_df()
            results_seed[syn_data] = scores_df["Score"].values.tolist()
        except Exception as err:
            print(f"Skipping {syn_data}: could not read: {err}")
    results_all[seed] = results_seed    

In [2]:
def calculate_aitchison_metrics(df_orig, df_synth):
    """
    Tính toán khoảng cách Aitchison cho từng Cell Type.
    Giả định: df_orig và df_synth có cùng index (Samples) và columns (Cell Types).
    """
    # 1. Đảm bảo các cột và hàng khớp nhau
    common_cols = df_orig.columns.intersection(df_synth.columns)
    df_orig = df_orig[common_cols]
    df_synth = df_synth[common_cols]

    # 2. Xử lý giá trị 0 (Multiplicative Replacement)
    # CIBERSORTx thường có số 0, phép này thay thế 0 bằng giá trị nhỏ 
    # nhưng vẫn giữ nguyên tổng bằng 1.
    orig_filled = multiplicative_replacement(df_orig.values)
    synth_filled = multiplicative_replacement(df_synth.values)

    # 3. Biến đổi CLR (Centered Log-Ratio)
    clr_orig = clr(orig_filled)
    clr_synth = clr(synth_filled)

    # Chuyển về DataFrame để dễ thao tác
    clr_orig_df = pd.DataFrame(clr_orig, index=df_orig.index, columns=df_orig.columns)
    clr_synth_df = pd.DataFrame(clr_synth, index=df_synth.index, columns=df_synth.columns)

    # 4. Tính Aitchison Distance cho từng Cell Type (theo cột)
    # Khoảng cách giữa 2 phân phối của cùng một loại tế bào qua các mẫu
    diff_sq = (clr_orig_df - clr_synth_df) ** 2
    aitchison_per_cell = np.sqrt(np.sum(diff_sq, axis=0))
    return aitchison_per_cell.sort_values()

results = calculate_aitchison_metrics(ori, syn)

/binary/miniforge3/envs/synthetic_data/lib/python3.9/site-packages/skbio/util/_warning.py:42: DeprecationWarning: `multiplicative_replacement` was renamed to `multi_replace` in 0.6.0. The old name is kept as an alias but is deprecated.
  warn(message, warning)


In [3]:
results_aitchison = {}
seeds = [0,1,2,3,42]
for seed in seeds:
    print(f"---Seed {seed}---")
    syn_datas = [f"avatarsk5_{seed}",f"avatarsk10_{seed}",f"ctgan_{seed}",f"gaussiancopula_{seed}",f"synthpop_{seed}",f"tvae_{seed}"]
    results_seed = {}
    for syn_data in syn_datas:
        try:
            print(f"---Method {syn_data}---")
            syn_df = pd.read_csv(f'ResultsCibersortx/CIBERSORTx_{syn_data}_Results.csv')
            syn = syn_df.iloc[:,1:23]
            results = calculate_aitchison_metrics(ori, syn)
            scores = results.mean()
            results_seed[syn_data] =scores
        except Exception as err:
            print(f"Skipping {syn_data}: could not read: {err}")
    results_aitchison[seed] = results_seed    

---Seed 0---
---Method avatarsk5_0---
---Method avatarsk10_0---
---Method ctgan_0---
---Method gaussiancopula_0---
---Method synthpop_0---
---Method tvae_0---
---Seed 1---
---Method avatarsk5_1---
---Method avatarsk10_1---
---Method ctgan_1---
---Method gaussiancopula_1---
---Method synthpop_1---
---Method tvae_1---
---Seed 2---
---Method avatarsk5_2---
---Method avatarsk10_2---
---Method ctgan_2---
---Method gaussiancopula_2---
---Method synthpop_2---
---Method tvae_2---
---Seed 3---
---Method avatarsk5_3---
---Method avatarsk10_3---
---Method ctgan_3---
---Method gaussiancopula_3---
---Method synthpop_3---
---Method tvae_3---
---Seed 42---
---Method avatarsk5_42---
---Method avatarsk10_42---
---Method ctgan_42---
---Method gaussiancopula_42---
---Method synthpop_42---
---Method tvae_42---


In [12]:
from collections import defaultdict

scores_by_tool = defaultdict(list)

for seed_dict in results_aitchison.values():
    for key, score in seed_dict.items():
        tool = key.rsplit("_", 1)[0]   # bỏ seed
        scores_by_tool[tool].append(score)

# Tính average
avg_scores = {tool: np.mean(scores) for tool, scores in scores_by_tool.items()}

avg_scores

{'avatarsk5': 15.737118926820767,
 'avatarsk10': 15.652759036969433,
 'ctgan': 21.649582399549416,
 'gaussiancopula': 20.17148237572571,
 'synthpop': 18.697653473975606,
 'tvae': 18.935353989884046}

In [13]:
from scipy.stats import wasserstein_distance
# Giả sử bạn đã import clr và multiplicative_replacement từ skbio.stats.composition

def calculate_was_dis(df_orig, df_synth):
    """
    Tính toán Wasserstein Distance cho từng Cell Type trong không gian CLR.
    """
    # 1. Đảm bảo các cột khớp nhau
    common_cols = df_orig.columns.intersection(df_synth.columns)
    df_orig = df_orig[common_cols]
    df_synth = df_synth[common_cols]

    # 2. Xử lý giá trị 0 (Sử dụng skbio)
    orig_filled = multiplicative_replacement(df_orig.values)
    synth_filled = multiplicative_replacement(df_synth.values)

    # 3. Biến đổi CLR (Centered Log-Ratio)
    clr_orig = clr(orig_filled)
    clr_synth = clr(synth_filled)

    # Chuyển về DataFrame để dễ truy xuất theo tên Cell Type
    clr_orig_df = pd.DataFrame(clr_orig, index=df_orig.index, columns=df_orig.columns)
    clr_synth_df = pd.DataFrame(clr_synth, index=df_synth.index, columns=df_synth.columns)

    # 4. TÍNH WASSERSTEIN DISTANCE CHO TỪNG CELL TYPE
    was_dists = {}
    for cell_type in common_cols:
        # Lấy phân phối của cell type đó qua các mẫu (hàng)
        u_dist = clr_orig_df[cell_type].values
        v_dist = clr_synth_df[cell_type].values
        
        # Tính Wasserstein distance (Earth Mover's Distance)
        was_dists[cell_type] = wasserstein_distance(u_dist, v_dist)

    # Chuyển kết quả sang Series và sắp xếp
    results = pd.Series(was_dists).sort_values()
    
    return results

# Thực thi
results = calculate_was_dis(ori, syn)

In [6]:
results_was_distance = {}
seeds = [0,1,2,3,42]
for seed in seeds:
    print(f"---Seed {seed}---")
    syn_datas = [f"avatarsk5_{seed}",f"avatarsk10_{seed}",f"ctgan_{seed}",f"gaussiancopula_{seed}",f"synthpop_{seed}",f"tvae_{seed}"]
    results_seed = {}
    for syn_data in syn_datas:
        try:
            print(f"---Method {syn_data}---")
            syn_df = pd.read_csv(f'ResultsCibersortx/CIBERSORTx_{syn_data}_Results.csv')
            syn = syn_df.iloc[:,1:23]
            results = calculate_was_dis(ori, syn)
            scores = results.mean()
            results_seed[syn_data] =scores
        except Exception as err:
            print(f"Skipping {syn_data}: could not read: {err}")
    results_was_distance[seed] = results_seed    

---Seed 0---
---Method avatarsk5_0---
---Method avatarsk10_0---
---Method ctgan_0---
---Method gaussiancopula_0---
---Method synthpop_0---
---Method tvae_0---
---Seed 1---
---Method avatarsk5_1---
---Method avatarsk10_1---
---Method ctgan_1---
---Method gaussiancopula_1---
---Method synthpop_1---
---Method tvae_1---
---Seed 2---
---Method avatarsk5_2---
---Method avatarsk10_2---
---Method ctgan_2---
---Method gaussiancopula_2---
---Method synthpop_2---
---Method tvae_2---
---Seed 3---
---Method avatarsk5_3---
---Method avatarsk10_3---
---Method ctgan_3---
---Method gaussiancopula_3---
---Method synthpop_3---
---Method tvae_3---
---Seed 42---
---Method avatarsk5_42---
---Method avatarsk10_42---
---Method ctgan_42---
---Method gaussiancopula_42---
---Method synthpop_42---
---Method tvae_42---


In [11]:
from collections import defaultdict

scores_by_tool = defaultdict(list)

for seed_dict in results_was_distance.values():
    for key, score in seed_dict.items():
        tool = key.rsplit("_", 1)[0]   # bỏ seed
        scores_by_tool[tool].append(score)

# Tính average
avg_scores = {tool: np.mean(scores) for tool, scores in scores_by_tool.items()}

avg_scores

{'avatarsk5': 0.26291593282514075,
 'avatarsk10': 0.3041833092742901,
 'ctgan': 0.49611464955678625,
 'gaussiancopula': 0.2591854061553926,
 'synthpop': 0.19551448478227162,
 'tvae': 0.3338604808000184}

In [14]:
import sys
parent_path = '/mnt/digphat/syntheticDataBenchmark/Chuong_pipeline'
sys.path.append(parent_path)
from SynOmics.metrics.fidelity.UnivariateSimilarity import UnivariateSimilarity
from SynOmics.processing.metadata import MetaData

In [15]:
def log_center(df):
    orig_filled = multiplicative_replacement(df.values)
    clr_orig = clr(orig_filled)
    
    clr_orig_df = pd.DataFrame(clr_orig, index=df.index, columns=df.columns)

    return clr_orig_df



clr_ori = log_center(ori)


metadata = MetaData.get_metadata(data = clr_ori, 
                                 threshold_unique_values = 10, 
                                    ordinal_features = None)

results_all = {}
seeds = [0,1,2,3,42]
for seed in seeds:
    print(f"---Seed {seed}---")
    syn_datas = [f"avatarsk5_{seed}",f"avatarsk10_{seed}",f"ctgan_{seed}",f"gaussiancopula_{seed}",f"synthpop_{seed}",f"tvae_{seed}"]
    results_seed = {}
    for syn_data in syn_datas:
        try:
            print(f"---Method {syn_data}---")
            syn_df = pd.read_csv(f'ResultsCibersortx/CIBERSORTx_{syn_data}_Results.csv')
            syn = syn_df.iloc[:,1:23]
            clr_syn = log_center(syn)
            uni_comparison = UnivariateSimilarity(output_dir = f"KSStatistic/Seed_{seed}", logger_name = syn_data)
            scores = uni_comparison.get_univariate_score(
                                        original_data = clr_ori, 
                                        synthetic_data=clr_syn, 
                                        metadata=metadata)
            
            scores_df = uni_comparison.get_detail_df()
            results_seed[syn_data] = scores_df["Score"].values.tolist()
        except Exception as err:
            print(f"Skipping {syn_data}: could not read: {err}")
    results_all[seed] = results_seed    

/mnt/digphat/syntheticDataBenchmark/Chuong_pipeline/SynOmics/processing/metadata.py:43: DeprecationWarning: is_categorical_dtype is deprecated and will be removed in a future version. Use isinstance(dtype, pd.CategoricalDtype) instead
  if pd.api.types.is_object_dtype(data[col]) or pd.api.types.is_categorical_dtype(data[col]):


---Seed 0---
---Method avatarsk5_0---
2026-01-26 17:07:21 - DEBUG - Standalone logger initialized successfully.
2026-01-26 17:07:23 - INFO - Starting univariate similarity computation.


100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 22/22 [00:00<00:00, 574.86it/s]


2026-01-26 17:07:25 - INFO - Univariate similarity score: 0.861244019138756
2026-01-26 17:07:28 - INFO - Details DataFrame saved to KSStatistic/Seed_0/Detail_score_avatarsk5_0.csv
2026-01-26 17:07:34 - INFO - Histogram figure saved to KSStatistic/Seed_0/avatarsk5_0.png
---Method avatarsk10_0---
2026-01-26 17:07:36 - DEBUG - Standalone logger initialized successfully.
2026-01-26 17:07:37 - INFO - Starting univariate similarity computation.


100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 22/22 [00:00<00:00, 583.02it/s]


...

2026-01-26 17:11:42 - INFO - Univariate similarity score: 0.8615430622009569
2026-01-26 17:11:42 - INFO - Details DataFrame saved to KSStatistic/Seed_42/Detail_score_synthpop_42.csv


2026-01-26 17:11:45 - INFO - Histogram figure saved to KSStatistic/Seed_42/synthpop_42.png
---Method tvae_42---
2026-01-26 17:11:45 - DEBUG - Standalone logger initialized successfully.
2026-01-26 17:11:45 - INFO - Starting univariate similarity computation.


100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 22/22 [00:00<00:00, 576.64it/s]

2026-01-26 17:11:45 - INFO - Univariate similarity score: 0.8145933014354068


2026-01-26 17:11:47 - INFO - Details DataFrame saved to KSStatistic/Seed_42/Detail_score_tvae_42.csv
2026-01-26 17:11:49 - INFO - Histogram figure saved to KSStatistic/Seed_42/tvae_42.png


In [16]:
scores_by_tool = defaultdict(list)

for seed_dict in results_all.values():
    for key, score in seed_dict.items():
        tool = key.rsplit("_", 1)[0]   # bỏ seed
        scores_by_tool[tool].append(score)

# Tính average
avg_scores = {tool: np.mean(scores) for tool, scores in scores_by_tool.items()}

avg_scores

{'avatarsk5': 0.8542464114832535,
 'avatarsk10': 0.8326555023923445,
 'ctgan': 0.7651913875598086,
 'gaussiancopula': 0.8566387559808613,
 'synthpop': 0.8714114832535885,
 'tvae': 0.8131578947368423}